In [9]:
import pandas as pd
import numpy as np
import zmq
import json
import time
from datetime import datetime

## For Id=4020332650

In [10]:
# Load the heart_rate from dataset
df = pd.read_csv('4020332650_heart_rate.csv', parse_dates=['Time'])

In [11]:
df.head()

,Time,Value
0,2016-04-01 01:03:00,68.684211
1,2016-04-01 01:04:00,64.518519
2,2016-04-01 01:05:00,68.423077
3,2016-04-01 01:06:00,67.655172
4,2016-04-01 01:07:00,67.240000


In [12]:
df['Time'] = pd.to_datetime(df['Time'])
df.set_index('Time', inplace=True)

In [13]:
# Ensure entire column is int64 (in case originals weren't)
df['Value'] = df['Value'].astype('int64')

In [14]:
df.head()

,Value
Time,
2016-04-01 01:03:00,68
2016-04-01 01:04:00,64
2016-04-01 01:05:00,68
2016-04-01 01:06:00,67
2016-04-01 01:07:00,67


In [15]:
# Step 2: Set up ZeroMQ publisher
context = zmq.Context()
socket = context.socket(zmq.PUB)
socket.bind("tcp://*:5556")  # Publisher binds to port 5556

print("Publisher starting... Sending heart rate data every 60 seconds.")
print(f"Total data points: {len(df)}")
print("Press Ctrl+C to stop early.\n")

Publisher starting... Sending heart rate data every 60 seconds.
Total data points: 16097
Press Ctrl+C to stop early.



In [16]:
# Sending every minutes data row by row with 1-second delays
try:
    for idx, row in df.iterrows():
        # Prepare message as JSON
        message = {
            'datetime': idx.isoformat(),  # Use ISO format for easy parsing
            'hr': float(row['Value'])     # Ensure HR is float
        }
        socket.send_string(json.dumps(message))
        
        print(f"Published at {datetime.now().strftime('%H:%M:%S')}: {message}")

        # Wait 0.1 seconds (we taking 0.1 seconds as 1 minute) before next publish
        time.sleep(0.1)
        
except KeyboardInterrupt:
    print("\nPublisher stopped by user.")
finally:
    socket.close()
    context.term()
    print("Publisher closed.")

Published at 22:41:41: {'datetime': '2016-04-01T01:03:00', 'hr': 68.0}
Published at 22:41:41: {'datetime': '2016-04-01T01:04:00', 'hr': 64.0}
Published at 22:41:41: {'datetime': '2016-04-01T01:05:00', 'hr': 68.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:06:00', 'hr': 67.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:07:00', 'hr': 67.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:08:00', 'hr': 67.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:09:00', 'hr': 67.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:10:00', 'hr': 68.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:11:00', 'hr': 69.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:12:00', 'hr': 69.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:13:00', 'hr': 71.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:14:00', 'hr': 71.0}
Published at 22:41:42: {'datetime': '2016-04-01T01:15:00', 'hr': 71.0}
Published at 22:41:43: {'datetime': '2016-04-01T01:16:00', 'hr': 71.0}
Publis